In [7]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split

# Directory where the image folders are located
image_dir = r"E:\lectures_and_books\Subjects\deep\ResNet\archive\HandGesture\images"

# Get all class folders (which are the subfolders)
class_folders = os.listdir(image_dir)

# Initialize lists to hold the paths and labels
image_paths = []
labels = []

# Loop through each class folder and collect image paths and labels
for class_name in class_folders:
    class_folder = os.path.join(image_dir, class_name)
    if os.path.isdir(class_folder):
        for image_file in os.listdir(class_folder):
            # Get the full image path
            image_paths.append(os.path.join(class_folder, image_file))
            # Use the class name as the label
            labels.append(class_name)

# Create a DataFrame with image paths and labels
df = pd.DataFrame({
    'image_path': image_paths,
    'label': labels
})

# Shuffle the DataFrame to randomize the data
df = df.sample(frac=1).reset_index(drop=True)

# Split the data into training and testing sets (80% train, 20% test)
train_df, test_df = train_test_split(df, test_size=0.2, stratify=df['label'])

print(f"Training set size: {len(train_df)}")
print(f"Test set size: {len(test_df)}")

# Optional: You can save these splits as CSV files if you want
train_df.to_csv("train_split.csv", index=False)
test_df.to_csv("test_split.csv", index=False)


Training set size: 4194
Test set size: 1049


In [19]:
import tensorflow as tf
from tensorflow.keras import layers, models, Input
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ModelCheckpoint
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split
from tensorflow import keras

In [9]:
def identity_block(X, filters, kernel_size):
    #Kernel size for the middle convolution
    F1, F2, F3 = filters #Number of Filters for the convolution layers

    # Save the input value so that I can use it while adding to f(x) before ReLU
    X_shortcut = X

    # First component of the main path
    X = layers.Conv2D(F1, (1, 1), strides=(1, 1), padding='valid')(X)
    X = layers.BatchNormalization()(X)
    X = layers.Activation('relu')(X)

    # Second component of the main path
    X = layers.Conv2D(F2, kernel_size, strides=(1, 1), padding='same')(X)
    X = layers.BatchNormalization()(X)
    X = layers.Activation('relu')(X)

    # Third component of the main path
    X = layers.Conv2D(F3, (1, 1), strides=(1, 1), padding='valid')(X)
    X = layers.BatchNormalization()(X)

    # Add shortcut value to the main path
    X = layers.Add()([X, X_shortcut])
    X = layers.Activation('relu')(X)

    return X


In [10]:

def convolutional_block(X, filters, kernel_size, strides=(2, 2)):
    F1, F2, F3 = filters

    # Save the input value
    X_shortcut = X

    # First component of the main path
    X = layers.Conv2D(F1, (1, 1), strides=strides, padding='valid')(X)
    X = layers.BatchNormalization()(X)
    X = layers.Activation('relu')(X)

    # Second component of the main path
    X = layers.Conv2D(F2, kernel_size, strides=(1, 1), padding='same')(X)
    X = layers.BatchNormalization()(X)
    X = layers.Activation('relu')(X)

    # Third component of the main path
    X = layers.Conv2D(F3, (1, 1), strides=(1, 1), padding='valid')(X)
    X = layers.BatchNormalization()(X)

    # Shortcut path
    X_shortcut = layers.Conv2D(F3, (1, 1), strides=strides, padding='valid')(X_shortcut)
    X_shortcut = layers.BatchNormalization()(X_shortcut)

    # Add shortcut value to the main path
    X = layers.Add()([X, X_shortcut])
    X = layers.Activation('relu')(X)

    return X


In [11]:

def ResNet50(input_shape=(224, 224, 3), classes=10):
    X_input = Input(input_shape)

    # Zero-padding
    X = layers.ZeroPadding2D((3, 3))(X_input)

    # Stage 1
    X = layers.Conv2D(64, (7, 7), strides=(2, 2), padding='valid')(X)
    X = layers.BatchNormalization()(X)
    X = layers.Activation('relu')(X)
    X = layers.MaxPooling2D((3, 3), strides=(2, 2), padding='same')(X)

    # Stage 2
    X = convolutional_block(X, filters=[64, 64, 256], kernel_size=3, strides=(1, 1))
    X = identity_block(X, filters=[64, 64, 256], kernel_size=3)
    X = identity_block(X, filters=[64, 64, 256], kernel_size=3)

    # Stage 3
    X = convolutional_block(X, filters=[128, 128, 512], kernel_size=3)
    X = identity_block(X, filters=[128, 128, 512], kernel_size=3)
    X = identity_block(X, filters=[128, 128, 512], kernel_size=3)
    X = identity_block(X, filters=[128, 128, 512], kernel_size=3)

    # Stage 4
    X = convolutional_block(X, filters=[256, 256, 1024], kernel_size=3)
    X = identity_block(X, filters=[256, 256, 1024], kernel_size=3)
    X = identity_block(X, filters=[256, 256, 1024], kernel_size=3)
    X = identity_block(X, filters=[256, 256, 1024], kernel_size=3)
    X = identity_block(X, filters=[256, 256, 1024], kernel_size=3)
    X = identity_block(X, filters=[256, 256, 1024], kernel_size=3)

    # Stage 5
    X = convolutional_block(X, filters=[512, 512, 2048], kernel_size=3)
    X = identity_block(X, filters=[512, 512, 2048], kernel_size=3)
    X = identity_block(X, filters=[512, 512, 2048], kernel_size=3)

    # Average pooling
    X = layers.AveragePooling2D((2, 2))(X)

    # Output layer
    X = layers.Flatten()(X)
    X = layers.Dense(classes, activation='softmax')(X)

    # Create model
    model = models.Model(inputs=X_input, outputs=X, name='ResNet50')

    return model




In [17]:
# Create ImageDataGenerators for training and testing
train_datagen = ImageDataGenerator(
    rescale=1./255,  # Normalize pixel values to [0, 1]
    rotation_range=30,  # Optional: Data augmentation for training
    width_shift_range=0.2,  # Horizontal shift
    height_shift_range=0.2,  # Vertical shift
    shear_range=0.2,  # Shear transformation
    zoom_range=0.2,  # Zoom in/out
    horizontal_flip=True,  # Flip images horizontally
    fill_mode='nearest'
)

test_datagen = ImageDataGenerator(rescale=1./255)  # Only rescaling for the test set

# Directory structure for ImageDataGenerator
# The directories should be organized as:
# image_dir/
# ├── call_me/
# ├── fingers_crossed/
# ├── okay/
# ├── paper/
# ├── peace/
# ├── rock/
# ├── rock_on/
# ├── scissor/
# ├── thumbs/
# └── up/

# Apply ImageDataGenerator to load images in batches
train_generator = train_datagen.flow_from_directory(
    image_dir,
    target_size=(224, 224),  # Resize images to match the input size of the model
    batch_size=32,  # Number of images per batch
    class_mode='categorical'  # Use categorical labels (one-hot encoded)
)

test_generator = test_datagen.flow_from_directory(
    image_dir,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical'
)

# Instantiate the ResNet50 model
model = ResNet50(input_shape=(224, 224, 3), classes=10)

# Compile the model
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Display the model summary
model.summary()


Found 5243 images belonging to 10 classes.
Found 5243 images belonging to 10 classes.


Model: "ResNet50"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_3       │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ zero_padding2d_3    │ (None, 230, 230,  │          0 │ input_layer_3[0]… │
│ (ZeroPadding2D)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_159 (Conv2D) │ (None, 112, 112,  │      9,472 │ zero_padding2d_3… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 112, 112,  │        256 │ conv2d_159[0][0]  │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_147      │ (None, 112, 112,  │          0 │ batch_normalizat… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_3     │ (None, 56, 56,    │          0 │ activation_147[0… │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_160 (Conv2D) │ (None, 56, 56,    │      4,160 │ max_pooling2d_3[… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 56, 56,    │        256 │ conv2d_160[0][0]  │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_148      │ (None, 56, 56,    │          0 │ batch_normalizat… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_161 (Conv2D) │ (None, 56, 56,    │     36,928 │ activation_148[0… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 56, 56,    │        256 │ conv2d_161[0][0]  │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_149      │ (None, 56, 56,    │          0 │ batch_normalizat… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_162 (Conv2D) │ (None, 56, 56,    │     16,640 │ activation_149[0… │
│                     │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_163 (Conv2D) │ (None, 56, 56,    │     16,640 │ max_pooling2d_3[… │
│                     │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 56, 56,    │      1,024 │ conv2d_162[0][0]  │
│ (BatchNormalizatio… │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 56, 56,    │      1,024 │ conv2d_163[0][0]  │
│ (BatchNormalizatio… │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_48 (Add)        │ (None, 56, 56,    │          0 │ batch_normalizat

 Total params: 23,772,042 (90.68 MB)

 Trainable params: 23,718,922 (90.48 MB)

 Non-trainable params: 53,120 (207.50 KB)

In [21]:

# Train the model using the data generators
model.fit(
    train_generator,  # Train using the training generator
    epochs=10,  # Number of epochs to train for
    validation_data=test_generator,  # Use test data for validation
    validation_steps=len(test_generator),  # Number of steps in validation
    steps_per_epoch=len(train_generator),  # Number of steps per epoch
    callbacks=[ModelCheckpoint("resnet50.keras", save_best_only=True)]  # Optional: save best model
)

# Evaluate the model on the test set
test_loss, test_acc = model.evaluate(test_generator, steps=len(test_generator))
print(f"Test Loss: {test_loss}")
print(f"Test Accuracy: {test_acc}")

c:\Users\LENOVO\AppData\Local\Programs\Python\Python39\lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/10
164/164 ━━━━━━━━━━━━━━━━━━━━ 0s 30s/step - accuracy: 0.2302 - loss: 4.8680 

KeyboardInterrupt: 

In [12]:

# Instantiate and compile the model
model = ResNet50(input_shape=(224, 224, 3), classes=10)
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Display the model summary
model.summary()


Model: "ResNet50"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ zero_padding2d_1    │ (None, 230, 230,  │          0 │ input_layer_1[0]… │
│ (ZeroPadding2D)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_53 (Conv2D)  │ (None, 112, 112,  │      9,472 │ zero_padding2d_1… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 112, 112,  │        256 │ conv2d_53[0][0]   │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_49       │ (None, 112, 112,  │          0 │ batch_normalizat… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_1     │ (None, 56, 56,    │          0 │ activation_49[0]… │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_54 (Conv2D)  │ (None, 56, 56,    │      4,160 │ max_pooling2d_1[… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 56, 56,    │        256 │ conv2d_54[0][0]   │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_50       │ (None, 56, 56,    │          0 │ batch_normalizat… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_55 (Conv2D)  │ (None, 56, 56,    │     36,928 │ activation_50[0]… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 56, 56,    │        256 │ conv2d_55[0][0]   │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_51       │ (None, 56, 56,    │          0 │ batch_normalizat… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_56 (Conv2D)  │ (None, 56, 56,    │     16,640 │ activation_51[0]… │
│                     │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_57 (Conv2D)  │ (None, 56, 56,    │     16,640 │ max_pooling2d_1[… │
│                     │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 56, 56,    │      1,024 │ conv2d_56[0][0]   │
│ (BatchNormalizatio… │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 56, 56,    │      1,024 │ conv2d_57[0][0]   │
│ (BatchNormalizatio… │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_16 (Add)        │ (None, 56, 56,    │          0 │ batch_normalizat

 Total params: 23,772,042 (90.68 MB)

 Trainable params: 23,718,922 (90.48 MB)

 Non-trainable params: 53,120 (207.50 KB)